#  Installation

In [ ]:
%%capture
! pip install vllm langchain-openai langchain_google_genai langchain pandas

# Imports


In [ ]:

import pandas as pd

# LLM Providers
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompting
from langchain_core.prompts import PromptTemplate


# Load data


In [ ]:
df = pd.read_csv("data\wiki_movie_plots_deduped_with_summaries.csv").sample(1000)
df_sample=df.sample(10)


# Models

## Local Model

In [ ]:
# start local vLLM server 
!nohup vllm serve "Qwen/Qwen3-4B-Instruct-2507"    --max_model_len 15000 &

In [ ]:
# connect to local vLLM server via LangChain 
local_llm = ChatOpenAI(
    model="Qwen/Qwen3-4B-Instruct-2507",
    openai_api_key="EMPTY",
    openai_api_base="http://localhost:8000/v1",
)



nohup: appending output to 'nohup.out'


## Api Model

In [ ]:

llm_api = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key="your_google_api_key_here",
    )


# Summarization Pipeline

In [ ]:
  # Define prompt template
prompt = PromptTemplate(
        input_variables=["plot"],
        template=(
            "Summarize the following movie plot in 2-3 sentences without spoilers:\n\n"
            "{plot}\n\nSummary:"
        ),
    )

In [ ]:
local_chain = prompt | local_llm
api_chain = prompt | llm_api


## API Model

In [ ]:
apI_summaries = []
for plot in df_sample["Plot"]:
  summary = api_chain.invoke({"plot": plot})
  apI_summaries.append(summary.content)

df_sample["api_summary"] = apI_summaries

## Local Model

In [ ]:
local_summaries = []
for plot in df_sample["Plot"]:
  summary = local_chain.invoke({"plot": plot})
  local_summaries.append(summary.content)

df_sample["local_summary"] = local_summaries

# Result

In [ ]:
for idx, row in df_sample.iterrows():
    print(f"Movie : { row["Title"]}")
    print("-" * 80)

    print("Original Plot:")
    print(row["Plot"])
    print("\n")

    print(" Summary using gemini api:")
    print(row["api_summary"])
    print("\n")

    print(" Summary using lcoal model:")
    print(row["local_summary"])
    print("\n")


Movie : Irham Dmoo`i (Pity My Tears)
--------------------------------------------------------------------------------
Original Plot:
Faten Hamama plays Amal, whose father loses most of his money and almost goes bankrupt. She gets abandoned by her fiancé. The owner of a nearby factory steps up and offers to help her father until his conditions get better. Amal marries this man, but their life together as a couple turns out to be boring. Amal, who wasn't very excited about their relationship initially, slowly discovers her husband's good qualities and their life turns into a happy one.


 Summary using gemini api:
A young woman faces hardship when her father's business struggles and her fiancé abandons her. She marries a factory owner who offers assistance, but their initial life together lacks excitement. Over time, she comes to appreciate her husband's virtues, leading to a fulfilling and happy marriage.


 Summary using lcoal model:
Amal, after losing her father's fortune and being ab